In [ ]:
import re
import tempfile
import gradio as gr
from openai import OpenAI
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

In [ ]:
openai = OpenAI()
groq = Groq()

In [ ]:
agentic_ai_expert_system_prompt = """
You are an Agentic AI Expert Tutor with deep expertise in AI agents,
LLMs, LangGraph, CrewAI, AutoGen, OpenAI Agents SDK,
PydanticAI, DSPy, Semantic Kernel, MCP and production-grade
agent architectures.

Answer every question accurately and teach the user.

Provide:

- Beginner explanation
- Technical explanation
- Practical examples
- Code snippets
- Best practices
- Trade-offs
- Architecture guidance
- Debugging tips

Never hallucinate APIs.
"""

def agentic_ai_expert_user_prompt(question):

    return f"""
I want to learn Agentic AI professionally.

For every answer:

1. Explain simply.
2. Explain technically.
3. Give production insights.
4. Give code examples.
5. Mention trade-offs.
6. Mention best practices.

Question:

{question}
"""

In [ ]:
chemistry_expert_system_prompt = """
You are a Chemistry Expert Tutor with deep expertise in Organic, Inorganic, Physical, Analytical, Computational, Medicinal, Industrial, and Materials Chemistry, capable of answering questions from high-school to PhD and research level.
Provide scientifically accurate, evidence-based explanations, adapting the depth and complexity of answers to the user's knowledge level.
When appropriate, explain reaction mechanisms, molecular structures, thermodynamics, kinetics, spectroscopy, calculations, and laboratory techniques step-by-step.
Include chemical equations, reaction pathways, examples, real-world applications, and problem-solving methods to build deep understanding.
Never fabricate chemical facts, data, mechanisms, or safety information; explicitly state uncertainty and prioritize scientific accuracy and safe laboratory practices.
"""

def chemistry_expert_user_prompt(question):
    user_prompt = f"""
    I want to learn Chemistry from basic to advanced level. For every question, explain clearly with step-by-step reasoning, include equations, mechanisms, and real-world applications where relevant. Adjust depth based on difficulty (beginner to PhD level). Provide examples, problem-solving methods, and when needed, link concepts across Physical, Organic, and Inorganic Chemistry. Keep explanations accurate, structured, and exam/research oriented. Give me a comprehensive answer to the following question: "{question}"
    """
    return user_prompt

In [ ]:
EXPERTS = {

    "🤖 Agentic AI Tutor": {
        "system": agentic_ai_expert_system_prompt,
        "user": agentic_ai_expert_user_prompt
    },

    "⚗️ Chemistry Tutor": {
        "system": chemistry_expert_system_prompt,
        "user": chemistry_expert_user_prompt
    }

}

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=message
    )

    temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    temp_audio.write(response.content)
    temp_audio.close()

    return temp_audio.name

In [ ]:
def stream_answer(question, expert, model):

    system_prompt = EXPERTS[expert]["system"]
    user_prompt = EXPERTS[expert]["user"](question)

    # OpenAI models
    if model in ["GPT-5", "GPT-4.1", "GPT-4o-mini"]:

        stream = openai.chat.completions.create(

            model=model.lower(),

            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],

            stream=True
        )

    # Groq models
    else:

        stream = groq.chat.completions.create(

            model=model,

            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],

            stream=True
        )

    answer = ""

    for chunk in stream:

        delta = chunk.choices[0].delta.content or ""

        answer += delta

        yield answer

In [ ]:
def calculator(expression):
    try:
        return str(eval(expression))
    except Exception:
        return "Invalid Expression"

In [ ]:
def chat(message, history, expert, model):

    history = history or []

    # Add user message
    history.append(
        {
            "role": "user",
            "content": message
        }
    )

    # Placeholder assistant message
    history.append(
        {
            "role": "assistant",
            "content": ""
        }
    )

    # Calculator Tool
    if re.fullmatch(r"[0-9+\-*/ ().]+", message):

        answer = f"🧮 **Calculator Tool Used**\n\n**Answer:** {calculator(message)}"

        history[-1]["content"] = answer

        audio = talker(answer)

        yield history, audio

        return

    # Stream LLM Response
    answer = ""

    for chunk in stream_answer(question=message, expert=expert, model=model):

        answer = chunk

        history[-1]["content"] = answer

        # Update text only while streaming
        yield history, None

    # Generate Audio After Streaming Completes
    audio = talker(answer)

    yield history, audio

In [ ]:
# Gradio UI
with gr.Blocks(title="🤖 Agentic AI/Chemistry Expert Tutor") as demo:
    
    expert = gr.Dropdown(
        choices=[
            "🤖 Agentic AI Tutor",
            "⚗️ Chemistry Tutor"
        ],
        value="🤖 Agentic AI Tutor",
        label="Choose Expert"
    )

    model = gr.Dropdown(
        choices=["GPT-5", "GPT-4.1", "GPT-4o-mini", "llama-3.3-70b-versatile"],
        value="GPT-5",
        label="Select Model"
    )

    chatbot = gr.Chatbot(
        height=550,
        type="messages",
        label="Conversation"
    )

    audio = gr.Audio(
        autoplay=True,
        label="Assistant Voice"
    )

    msg = gr.Textbox(
        placeholder="Ask me anything about Agentic AI...",
        label="Your Question"
    )

    clear = gr.Button("🗑️ Clear Chat")

    msg.submit(
        fn=chat,
        inputs=[msg, chatbot, expert, model],
        outputs=[chatbot, audio]
    ).then(
        lambda: "",
        None,
        msg
    )

    clear.click(
        lambda: ([], None),
        outputs=[chatbot, audio]
    )

demo.launch(inbrowser=True, auth=("zain", "root"))